In [6]:
import pandas as pd

In [7]:
df=pd.read_csv("datasets/customer_support_data.csv")

In [8]:
df.head()

,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,Assigned_Agent,Satisfaction_Score
0,TKT-100000,George Simon,lisastrickland@example.com,Hours of operation - Individual,"Hi Support, Where is your headquarters located...",General Inquiry,High,Web Form,2025-07-02,43,David Kim,5
1,TKT-100001,Scott Thompson,wevans@example.org,Data not syncing - Card,"Hi Support, The application crashes every time...",Technical,High,Chat,2025-06-28,41,Elena Rodriguez,5
2,TKT-100002,Jennifer Smith,oleonard@example.net,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...",Account,High,Web Form,2025-02-05,7,Anya Sharma,5
3,TKT-100003,Rachel Bullock,katherine67@example.net,Login failed - Let,"Hi Support, The dashboard is not loading any d...",Technical,Low,Web Form,2025-03-20,41,Anya Sharma,5
4,TKT-100004,Thomas Parks DDS,raykelsey@example.com,Refund status - Attention,"Hi Support, I have been trying to update my pa...",Billing,Medium,Email,2025-04-27,40,David Kim,5


In [9]:
df.columns.tolist()

['Ticket_ID',
 'Customer_Name',
 'Customer_Email',
 'Ticket_Subject',
 'Ticket_Description',
 'Issue_Category',
 'Priority_Level',
 'Ticket_Channel',
 'Submission_Date',
 'Resolution_Time_Hours',
 'Assigned_Agent',
 'Satisfaction_Score']

In [10]:
df.isnull().sum()

Ticket_ID                0
Customer_Name            0
Customer_Email           0
Ticket_Subject           0
Ticket_Description       0
Issue_Category           0
Priority_Level           0
Ticket_Channel           0
Submission_Date          0
Resolution_Time_Hours    0
Assigned_Agent           0
Satisfaction_Score       0
dtype: int64

In [11]:
print(df["Issue_Category"].value_counts())


Issue_Category
Technical          5918
Billing            5036
Account            4081
General Inquiry    3925
Fraud              1040
Name: count, dtype: int64


In [12]:
text_columns = ["Ticket_Subject", "Ticket_Description"]
target_column = "Issue_Category"

# Remove rows missing input text or the target label
df = df.dropna(subset=text_columns + [target_column]).copy()

In [13]:
for column in text_columns + [target_column]:
    df[column] = df[column].astype(str).str.strip()

In [14]:
df = df[
    df["Ticket_Subject"].ne("") &
    df["Ticket_Description"].ne("") &
    df["Issue_Category"].ne("")
].copy()

In [15]:
# Combine subject and description into one model input

df["input_text"] = (
    df["Ticket_Subject"] + " " + df["Ticket_Description"]
).str.replace(r"\s+", " ", regex=True).str.strip()

In [16]:
# Check whether duplicate text has conflicting labels
label_counts = df.groupby("input_text")["Issue_Category"].nunique()
conflicting_text_count = (label_counts > 1).sum()

In [17]:
print("Rows after cleaning:", len(df))
print("Unique issue categories:", df["Issue_Category"].nunique())
print("Duplicate text entries with conflicting labels:", conflicting_text_count)

Rows after cleaning: 20000
Unique issue categories: 5
Duplicate text entries with conflicting labels: 0


In [18]:
from sklearn.model_selection import train_test_split

In [19]:
X = df["input_text"]
y = df["Issue_Category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 16000
Testing rows: 4000


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [21]:
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=20_000
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training feature matrix:", X_train_tfidf.shape)
print("Testing feature matrix:", X_test_tfidf.shape)

Training feature matrix: (16000, 20000)
Testing feature matrix: (4000, 20000)


In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

In [23]:
model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_tfidf, y_train)

# Baseline: always predict the most common training category
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_tfidf, y_train)

print("Classifier trained.")


Classifier trained.


In [24]:
from sklearn.metrics import accuracy_score, classification_report

In [25]:
y_pred = model.predict(X_test_tfidf)
baseline_pred = baseline.predict(X_test_tfidf)

print(f"Model accuracy:    {accuracy_score(y_test, y_pred):.3f}")
print(f"Baseline accuracy: {accuracy_score(y_test, baseline_pred):.3f}")

print("\nPrecision, recall, and F1 by category:")
print(classification_report(y_test, y_pred, zero_division=0))

Model accuracy:    1.000
Baseline accuracy: 0.296

Precision, recall, and F1 by category:
                 precision    recall  f1-score   support

        Account       1.00      1.00      1.00       816
        Billing       1.00      1.00      1.00      1007
          Fraud       1.00      1.00      1.00       208
General Inquiry       1.00      1.00      1.00       785
      Technical       1.00      1.00      1.00      1184

       accuracy                           1.00      4000
      macro avg       1.00      1.00      1.00      4000
   weighted avg       1.00      1.00      1.00      4000



In [26]:
def classify_issue(subject, description, confidence_threshold=0.60):
    request_text = f"{subject} {description}".strip()
    request_features = tfidf.transform([request_text])

    probabilities = model.predict_proba(request_features)[0]
    best_index = probabilities.argmax()

    category = model.classes_[best_index]
    confidence = probabilities[best_index]

    if confidence < confidence_threshold:
        result = "Needs agent review"
    else:
        result = category

    return {
        "predicted_category": result,
        "model_category": category,
        "confidence": round(float(confidence), 3)
    }


# Example
classify_issue(
    subject="Unable to log in",
    description="The app rejects my password even after I reset it."
)

{'predicted_category': 'Account',
 'model_category': 'Account',
 'confidence': 0.802}

In [27]:
classify_issue(
    subject="Charged twice for subscription",
    description="My latest invoice shows two charges for the same monthly subscription. Please check and refund the duplicate payment."
)

{'predicted_category': 'Billing',
 'model_category': 'Billing',
 'confidence': 0.991}

In [28]:
classify_issue(
    subject="Suspicious login and account activity",
    description="I received a login alert from a location I do not recognize and noticed changes to my account. Please investigate."
)

{'predicted_category': 'Fraud', 'model_category': 'Fraud', 'confidence': 0.946}

In [29]:
classify_issue(
    subject="Where can I find the user guide?",
    description="I am looking for instructions on how to use the main features of the service."
)

{'predicted_category': 'Needs agent review',
 'model_category': 'General Inquiry',
 'confidence': 0.458}

In [30]:
classify_issue(
    subject="Application crashes when opening settings",
    description="Whenever I open the settings page, the application closes unexpectedly. This happens every time."
)

{'predicted_category': 'Technical',
 'model_category': 'Technical',
 'confidence': 0.975}

In [31]:
classify_issue(
    subject="Two-factor code is not arriving",
    description="I can enter my password, but the sign-in page asks for a verification code. No code arrives by SMS or email, so I cannot access the workspace."
)

{'predicted_category': 'Needs agent review',
 'model_category': 'Account',
 'confidence': 0.588}

In [32]:
import joblib

model_bundle = {
    "vectorizer": tfidf,
    "classifier": model,
    "categories": model.classes_.tolist()
}

joblib.dump(
    model_bundle,
    r"D:\1Customer_Service_Knowledge\backend\model\classification_model.joblib"
)

print("Model saved successfully.")

Model saved successfully.
